In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

LASTFM_API_KEY = os.getenv("LASTFM_API_KEY")
BASE_URL = "https://ws.audioscrobbler.com/2.0/"

In [2]:
import requests

def get_track_description(artist: str, track: str):
    params = {
        "method": "track.getInfo",
        "artist": artist,
        "track": track,
        "api_key": LASTFM_API_KEY,
        "format": "json",
        "autocorrect": 1,
    }

    r = requests.get(BASE_URL, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    track_data = data.get("track", {})

    return {
        "name": track_data.get("name"),
        "artist": track_data.get("artist", {}).get("name") if isinstance(track_data.get("artist"), dict) else track_data.get("artist"),
        "description": track_data.get("wiki", {}).get("summary"),
    }

In [3]:
get_track_description("AWOL", "Food")

{'name': 'Food', 'artist': 'AWOL', 'description': None}

In [4]:
import pandas as pd

def get_top_tracks_for_tag(tag: str, limit: int = 50, page: int = 1):
    params = {
        "method": "tag.getTopTracks",
        "tag": tag,
        "api_key": LASTFM_API_KEY,
        "format": "json",
        "limit": limit,
        "page": page,
    }

    r = requests.get(BASE_URL, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    tracks = data.get("tracks", {}).get("track", [])

    rows = []
    for t in tracks:
        rows.append({
            "tag": tag,
            "name": t.get("name"),
            "artist": t.get("artist", {}).get("name"),
        })

    return rows

In [5]:
tags = [
    # крупные жанры
    "rock", "pop", "hip-hop", "rap", "electronic", "dance",
    "indie", "alternative", "metal", "punk",
    "jazz", "blues", "soul", "rnb",
    "classical", "soundtrack", "ambient",
    "folk", "country", "singer-songwriter",
    # дополнительные популярные теги
    "chillout", "lo-fi", "house", "techno",
    "trance", "drum and bass", "reggae",
    "k-pop", "j-pop", "latin", "funk"
]

In [6]:
all_rows = []

for tag in tags:
    rows = get_top_tracks_for_tag(tag, limit=40, page=1)
    all_rows.extend(rows)

tracks_df = pd.DataFrame(all_rows)
tracks_df = tracks_df.dropna(subset=["artist", "name"])
tracks_df = tracks_df.drop_duplicates(subset=["artist", "name"]).reset_index(drop=True)

len(tracks_df), tracks_df.head()

(1133,
     tag                            name         artist
 0  rock               Sign of the Times   Harry Styles
 1  rock                        Everlong   Foo Fighters
 2  rock                            Iris  Goo Goo Dolls
 3  rock                  Still Into You       Paramore
 4  rock  Lover, You Should've Come Over   Jeff Buckley)

In [7]:
tracks_df.tag.value_counts()

tag
rock                 40
pop                  40
hip-hop              40
metal                40
jazz                 40
classical            40
ambient              40
lo-fi                40
drum and bass        40
reggae               40
j-pop                40
electronic           39
punk                 39
soundtrack           39
country              39
techno               38
k-pop                38
latin                38
blues                37
folk                 37
funk                 37
singer-songwriter    35
trance               35
indie                34
rnb                  33
chillout             33
soul                 31
house                31
dance                29
alternative          27
rap                  24
Name: count, dtype: int64

In [8]:
import time

def get_track_description_safe(artist: str, track: str):
    try:
        params = {
            "method": "track.getInfo",
            "artist": artist,
            "track": track,
            "api_key": LASTFM_API_KEY,
            "format": "json",
            "autocorrect": 1,
        }
        r = requests.get(BASE_URL, params=params, timeout=10)
        r.raise_for_status()
        data = r.json()
        track_data = data.get("track", {})

        description = track_data.get("wiki", {}).get("summary")
        name = track_data.get("name")
        artist_name = (
            track_data.get("artist", {}).get("name")
            if isinstance(track_data.get("artist"), dict)
            else track_data.get("artist")
        )

        return {
            "name": name or track,
            "artist": artist_name or artist,
            "description": description,
        }
    except Exception:
        return {
            "name": track,
            "artist": artist,
            "description": None,
        }

In [9]:
descriptions = []

for i, row in tracks_df.iterrows():
    artist = row["artist"]
    track = row["name"]

    res = get_track_description_safe(artist, track)
    descriptions.append(res["description"])

    # небольшой прогресс‑принт, можешь убрать
    if (i + 1) % 50 == 0:
        print(f"Processed {i + 1} / {len(tracks_df)}")
        time.sleep(0.5)  # пауза после каждых 50 запросов

tracks_df["description"] = descriptions

Processed 50 / 1133
Processed 100 / 1133
Processed 150 / 1133
Processed 200 / 1133
Processed 250 / 1133
Processed 300 / 1133
Processed 350 / 1133
Processed 400 / 1133
Processed 450 / 1133
Processed 500 / 1133
Processed 550 / 1133
Processed 600 / 1133
Processed 650 / 1133
Processed 700 / 1133
Processed 750 / 1133
Processed 800 / 1133
Processed 850 / 1133
Processed 900 / 1133
Processed 950 / 1133
Processed 1000 / 1133
Processed 1050 / 1133
Processed 1100 / 1133


In [10]:
num_with_desc = tracks_df["description"].notna().sum()
num_total = len(tracks_df)
num_with_desc, num_with_desc / num_total

(np.int64(909), np.float64(0.8022947925860547))

In [11]:
tracks_with_desc = tracks_df.dropna(subset=["description"]).reset_index(drop=True)

len(tracks_df), len(tracks_with_desc)

(1133, 909)

In [12]:
tracks_with_desc.tag.value_counts()

tag
rock                 40
pop                  40
hip-hop              40
electronic           39
metal                38
k-pop                37
funk                 35
folk                 34
country              34
punk                 33
rnb                  33
reggae               33
indie                32
singer-songwriter    31
ambient              30
dance                29
jazz                 29
soul                 29
alternative          27
house                26
blues                24
chillout             24
latin                24
rap                  23
classical            23
soundtrack           23
drum and bass        23
lo-fi                21
j-pop                20
techno               18
trance               17
Name: count, dtype: int64

In [ ]:
import os

os.makedirs("/data/processed", exist_ok=True)

cols_to_save = ["artist", "name", "tag", "description"]

tracks_with_desc[cols_to_save].to_parquet(
    "/data/processed/data_fm.parquet",
    index=False
)

Сохранено: data\processed\data_fm.parquet


In [18]:
from pathlib import Path

save_dir = Path("../data/processed")
save_dir.mkdir(parents=True, exist_ok=True)

tracks_with_desc[cols_to_save].to_parquet(
    save_dir / "data_fm.parquet",
    index=False
)
print("Сохранено в:", save_dir.resolve())

Сохранено в: K:\IMSH\music-recommendations-\data\processed
